# Political sycophancy: statistical analysis

New analyses of the supplied probability scores. Each biography condition is paired with its question's no-biography baseline. Both identities and repeated question texts stay together for inference.

Effects are record-weighted mean percentage-point changes and differences in **log-odds of mean agreement**, $\operatorname{logit}(\bar p_1)-\operatorname{logit}(\bar p_0)$, using natural logarithms. The latter requires no clipping here. It differs from averaging individual logit changes, examined below.

Two-sided tests and pointwise 95% confidence intervals use question-clustered standard errors, a $G/(G-1)$ correction and $t_{G-1}$ reference distribution. Log-odds standard errors use the delta method. Holm correction covers all tests within each displayed result family. Confidence intervals are not multiplicity-adjusted.

Distinct question texts are assumed independent. Unknown generation-batch dependence is not accounted for. These tests concern variation across the supplied synthetic prompts, not binomial response counts, repeat-query variability or effects in human populations.

In [1]:
import numpy as np
import pandas as pd
from scipy.special import logit
from scipy.stats import t
from IPython.display import display

pd.options.display.max_rows = 120
pd.options.display.float_format = '{:.3f}'.format
data = pd.read_csv('csv_data/pol_data_percent.csv')
scores = ['agree_A', 'agree_B', 'empty_A', 'empty_B', 'disagree_A', 'disagree_B']
assert data[scores].notna().all().all()
assert data[scores].ge(0).all().all() and data[scores].le(100).all().all()
assert data[['question', 'option_A', 'option_B', 'identityA', 'identityB', 'topic', 'model']].notna().all().all()
assert np.allclose(data.disagree_A, 100 - data.agree_A)
assert np.allclose(data.disagree_B, 100 - data.agree_B)

options = pd.concat([
    data[['model', 'question', 'topic', 'identityA', 'agree_A', 'empty_A', 'disagree_B']]
        .rename(columns={'identityA': 'identity', 'agree_A': 'aligned',
                         'empty_A': 'baseline', 'disagree_B': 'opposing'}),
    data[['model', 'question', 'topic', 'identityB', 'agree_B', 'empty_B', 'disagree_A']]
        .rename(columns={'identityB': 'identity', 'agree_B': 'aligned',
                         'empty_B': 'baseline', 'disagree_A': 'opposing'}),
], ignore_index=True)
options[['aligned', 'baseline', 'opposing']] /= 100
coverage = data.groupby('model').agg(records=('question', 'size'),
                                    question_texts=('question', 'nunique'))
coverage['identity_observations'] = 2 * coverage.records
coverage

,records,question_texts,identity_observations
model,,,
gpt_3_5,12924,10246,25848
gpt_4,13122,10409,26244


## Estimation

Opposing-biography scores follow the existing plots: one minus agreement with the other identity. Standard errors sum influence values within question text. Method references: [cluster covariance](https://www.statsmodels.org/stable/generated/statsmodels.stats.sandwich_covariance.cov_cluster.html) and [Holm correction](https://www.statsmodels.org/stable/generated/statsmodels.stats.multitest.multipletests.html).

In [2]:
def cluster_test(estimate, influence, questions):
    sums = pd.Series(influence).groupby(np.asarray(questions)).sum()
    n, g = len(influence), len(sums)
    if g < 2:
        raise ValueError('At least two question clusters are required.')
    se = np.sqrt(g / (g - 1) * np.square(sums).sum()) / n
    if se == 0:
        raise ValueError('The contrast has zero estimated standard error.')
    statistic = estimate / se
    margin = t.ppf(0.975, g - 1) * se
    return dict(n=n, questions=g, estimate=estimate, se=se,
                ci_low=estimate - margin, ci_high=estimate + margin,
                t=statistic, p_value=2 * t.sf(abs(statistic), g - 1))


def contrast(frame, columns, weights, scale):
    values = frame[columns].to_numpy()
    means = values.mean(axis=0)
    weights = np.asarray(weights)
    if scale == 'percentage_points':
        estimate = 100 * (means @ weights)
        influence = 100 * ((values - means) @ weights)
    elif scale == 'log_odds':
        if not ((means > 0) & (means < 1)).all():
            raise ValueError('Mean probabilities must lie strictly between 0 and 1.')
        estimate = logit(means) @ weights
        influence = (values - means) @ (weights / (means * (1 - means)))
    else:
        raise ValueError(scale)
    return cluster_test(estimate, influence, frame.question)


def holm(table):
    table = table.copy()
    p = table.p_value.to_numpy()
    order = np.argsort(p)
    adjusted = np.minimum(1, np.maximum.accumulate(p[order] * np.arange(len(p), 0, -1)))
    table.loc[table.index[order], 'p_holm'] = adjusted
    return table


def readable(table):
    table = table.copy()
    for column in ['p_value', 'p_holm']:
        if column in table:
            table[column] = table[column].map(lambda p: '<1e-300' if p < 1e-300 else f'{p:.3g}')
    return table


scales = ['percentage_points', 'log_odds']
shown = ['estimate', 'ci_low', 'ci_high', 'p_holm']

## Biography effects

Positive effects indicate greater agreement than the no-biography baseline. Holm families: 8 overall tests and 112 identity-specific tests, each including both models, both biography contrasts and both scales.

In [3]:
overall_rows, identity_rows = [], []
for model, frame in options.groupby('model'):
    for condition in ['aligned', 'opposing']:
        for scale in scales:
            metadata = dict(model=model, condition=condition, scale=scale)
            overall_rows.append({**metadata, **contrast(frame, [condition, 'baseline'], [1, -1], scale)})
            for identity, group in frame.groupby('identity'):
                identity_rows.append({**metadata, 'identity': identity,
                                      **contrast(group, [condition, 'baseline'], [1, -1], scale)})
overall = holm(pd.DataFrame(overall_rows))
by_identity = holm(pd.DataFrame(identity_rows))
display(readable(overall.set_index(['model', 'condition', 'scale'])[shown]))
display(readable(by_identity.query("condition == 'aligned'")
                 .set_index(['model', 'identity', 'scale'])[shown]))

estimate  ci_low  ci_high   p_holm
model   condition scale                                                
gpt_3_5 aligned   percentage_points    29.189  28.635   29.743  <1e-300
                  log_odds              1.318   1.286    1.350  <1e-300
        opposing  percentage_points   -27.311 -27.827  -26.796  <1e-300
                  log_odds             -1.243  -1.273   -1.212  <1e-300
gpt_4   aligned   percentage_points    25.017  24.451   25.584  <1e-300
                  log_odds              1.093   1.063    1.122  <1e-300
        opposing  percentage_points   -23.953 -24.540  -23.366  <1e-300
                  log_odds             -1.050  -1.081   -1.020  <1e-300

estimate  ci_low  ci_high  \
model   identity          scale                                          
gpt_3_5 authoritarians    percentage_points    39.749  36.962   42.537   
        capitalists       percentage_points    41.474  39.267   43.681   
        collectivists     percentage_points    14.955  13.352   16.558   
        conservatives     percentage_points    63.010  60.964   65.056   
        environmentalists percentage_points     8.224   6.589    9.859   
        individualists    percentage_points    53.531  51.259   55.802   
        industrialists    percentage_points    33.801  31.088   36.514   
        liberals          percentage_points     9.883   8.517   11.250   
        libertarians      percentage_points    32.911  30.269   35.553   
        progressives      percentage_points    10.302   8.744   11.861   
        secularists       percentage_points     7.060   5.964    8.156   
        socialists        percentage_points    14.089  12.536   15.642   
        theocrats         percentage_points    23.379  21.093   25.666   
        traditionalists   percentage_points    48.276  44.917   51.635   
        authoritarians    log_odds              1.795   1.668    1.921   
        capitalists       log_odds              1.970   1.848    2.092   
        collectivists     log_odds              2.445   2.150    2.740   
        conservatives     log_odds              3.229   3.069    3.389   
        environmentalists log_odds              1.166   0.918    1.414   
        individualists    log_odds              2.512   2.375    2.649   
        industrialists    log_odds              1.880   1.705    2.054   
        liberals          log_odds              1.584   1.354    1.814   
        libertarians      log_odds              1.870   1.731    2.009   
        progressives      log_odds              1.656   1.222    2.089   
        secularists       log_odds              0.784   0.650    0.918   
        socialists        log_odds              1.859   1.654    2.064   
        theocrats         log_odds              1.346   1.201    1.491   
        traditionalists   log_odds              2.419   2.205    2.634   
gpt_4   authoritarians    percentage_points    30.938  28.605   33.270   
        capitalists       percentage_points    41.426  39.256   43.597   
        collectivists     percentage_points    12.496  11.087   13.906   
        conservatives     percentage_points    58.637  56.541   60.733   
        environmentalists percentage_points     6.638   5.183    8.094   
        individualists    percentage_points    40.235  37.929   42.541   
        industrialists    percentage_points    25.232  22.782   27.682   
        liberals          percentage_points     6.050   5.095    7.006   
        libertarians      percentage_points    31.727  29.029   34.425   
        progressives      percentage_points     9.663   8.064   11.262   
        secularists       percentage_points     6.477   5.422    7.531   
        socialists        percentage_points    13.504  11.863   15.145   
        theocrats         percentage_points    15.559  13.770   17.349   
        traditionalists   percentage_points    41.695  38.119   45.272   
        authoritarians    log_odds              1.305   1.205    1.406   
        capitalists       log_odds              1.938   1.820    2.055   
        collectivists     log_odds              1.574   1.386    1.762   
        conservatives     log_odds              2.999   2.851    3.146   
        environmentalists log_odds              0.726   0.562    0.891   
        individualists    log_odds              1.940   1.815    2.065   
        industrialists    log_odds              1.384   1.230    1.539   
        liberals          log_odds              1.014   0.848    1.180   
        libertarians      log_odds              1.866   1.717    2.016   
        progressives      log_odds              1.462   1.139    1.785   
        secularists       log_odds              0

## Exploratory ideological comparison

The mapping below is an analytical convention, not an intrinsic classification of every identity. Authoritarians/libertarians are excluded. Each retained record contributes both a left-labelled and a right-labelled identity. Positive contrasts mean a larger aligned-biography shift for the right-labelled group. Holm correction covers four pooled tests. Pair-specific contrasts are also reported, with a separate family of 24 tests.

In [4]:
left_right = {
    'liberals': 'conservatives',
    'collectivists': 'individualists',
    'environmentalists': 'industrialists',
    'socialists': 'capitalists',
    'secularists': 'theocrats',
    'progressives': 'traditionalists',
}
display(pd.DataFrame(left_right.items(), columns=['Left label', 'Right label']))
lr = data.loc[data.identityA.isin(left_right)].copy()
assert (lr.identityA.map(left_right) == lr.identityB).all()
lr[scores] /= 100
columns = ['agree_B', 'empty_B', 'agree_A', 'empty_A']
weights = [1, -1, -1, 1]
lr_rows, pair_rows = [], []
for model, frame in lr.groupby('model'):
    for scale in scales:
        lr_rows.append(dict(model=model, scale=scale, **contrast(frame, columns, weights, scale)))
        for pair, group in frame.groupby(['identityA', 'identityB']):
            pair_rows.append(dict(model=model, left=pair[0], right=pair[1], scale=scale,
                                  **contrast(group, columns, weights, scale)))
ideology = holm(pd.DataFrame(lr_rows))
ideology_pairs = holm(pd.DataFrame(pair_rows))
display(readable(ideology.set_index(['model', 'scale'])[shown]))
display(readable(ideology_pairs.query("scale == 'log_odds'")
                 .set_index(['model', 'left', 'right'])[shown]))

,Left label,Right label
0,liberals,conservatives
1,collectivists,individualists
2,environmentalists,industrialists
3,socialists,capitalists
4,secularists,theocrats
5,progressives,traditionalists


estimate  ci_low  ci_high    p_holm
model   scale                                                 
gpt_3_5 percentage_points    34.000  32.587   35.412   <1e-300
        log_odds              0.701   0.586    0.817  3.12e-32
gpt_4   percentage_points    28.938  27.565   30.312   <1e-300
        log_odds              0.757   0.660    0.854  4.76e-52

estimate  ci_low  ci_high    p_holm
model   left              right                                               
gpt_3_5 collectivists     individualists      0.067  -0.258    0.392     0.721
        environmentalists industrialists      0.714   0.415    1.013  2.29e-05
        liberals          conservatives       1.644   1.365    1.924  1.17e-28
        progressives      traditionalists     0.764   0.378    1.149  0.000634
        secularists       theocrats           0.562   0.382    0.742  1.23e-08
        socialists        capitalists         0.112  -0.128    0.351     0.721
gpt_4   collectivists     individualists      0.366   0.149    0.583   0.00417
        environmentalists industrialists      0.658   0.439    0.877  4.55e-08
        liberals          conservatives       1.985   1.768    2.202  7.78e-65
        progressives      traditionalists     0.845   0.534    1.157  9.47e-07
        secularists       theocrats           0.266   0.110    0.422   0.00417
        socialists        capitalists         0.262   0.019    0.505     0.103

## Log-odds sensitivity

These are **mean individual logit changes**, a different estimand from the primary analysis. Clip individual probabilities to $[\epsilon,1-\epsilon]$ because some equal 1. The table checks three thresholds and reports descriptive effects, without additional significance tests.

In [5]:
sensitivity_rows = []
for epsilon in [1e-8, 1e-6, 1e-4]:
    frame = options.copy()
    frame['shift'] = logit(frame.aligned.clip(epsilon, 1 - epsilon)) - logit(frame.baseline.clip(epsilon, 1 - epsilon))
    for model, group in frame.groupby('model'):
        left = group.loc[group.identity.isin(left_right), 'shift'].mean()
        right = group.loc[group.identity.isin(left_right.values()), 'shift'].mean()
        sensitivity_rows.append(dict(model=model, epsilon=epsilon,
                                     mean_logit_change=group['shift'].mean(),
                                     left=left, right=right, right_minus_left=right - left))
sensitivity = pd.DataFrame(sensitivity_rows)
display(sensitivity.assign(epsilon=sensitivity.epsilon.map('{:.0e}'.format))
        .set_index(['model', 'epsilon']))

,,mean_logit_change,left,right,right_minus_left
model,epsilon,,,,
gpt_3_5,1e-08,5.607,3.014,7.900,4.886
gpt_4,1e-08,6.072,3.802,7.727,3.926
gpt_3_5,1e-06,5.607,3.014,7.900,4.886
gpt_4,1e-06,5.968,3.681,7.677,3.995
gpt_3_5,1e-04,5.327,2.929,7.411,4.481
gpt_4,1e-04,4.580,2.229,6.538,4.310
